# 📘 XGBoost Classification — Manual Working 

---

## 1️⃣ Dataset (Small Example)

| X | y |
|---|---|
| 1 | 0 |
| 2 | 0 |
| 3 | 1 |
| 4 | 1 |

---

## 2️⃣ Initial Prediction (Log-Odds Base Score)

$$
p = \frac{2}{4} = 0.5
$$

$$
\text{log-odds} = \log\left(\frac{p}{1-p}\right) = \log(1) = 0
$$

Initial prediction for all samples in log-odds space is **0**.

---

## 3️⃣ Convert to Probability (Sigmoid)

$$
p_i = \frac{1}{1 + e^{-0}} = 0.5
$$

All predicted probabilities \( p_i = 0.5 \).

---

## 4️⃣ Residuals for Samples

$$
r_i = y_i - p_i
$$

| X | y | \(p_i\) | Residual \(r_i\) |
|---|---|---------|------------------|
| 1 | 0 | 0.5     | -0.5             |
| 2 | 0 | 0.5     | -0.5             |
| 3 | 1 | 0.5     | +0.5             |
| 4 | 1 | 0.5     | +0.5             |

---

## 5️⃣ Similarity Score (SS) Formula

$$
SS = \frac{\left(\sum r_i\right)^2}{\sum p_i (1 - p_i) + \lambda}
$$

Where:
- \( r_i \) = residuals in the node  
- \( p_i \) = predicted probabilities in the node  
- \( \lambda \) = regularization parameter (assumed \( \lambda = 1 \))

---

## 6️⃣ Calculate SS for Parent Node (All samples)

$$
\sum r_i = (-0.5) + (-0.5) + 0.5 + 0.5 = 0
$$

$$
\sum p_i (1 - p_i) = 4 \times 0.5 \times (1 - 0.5) = 4 \times 0.25 = 1
$$

$$
SS_{parent} = \frac{0^2}{1 + 1} = 0
$$

---

## 7️⃣ Try Split: X ≤ 2


### Left Node (Samples \(X=1,2\))

- Residuals: -0.5, -0.5  
- Sum residuals = -1  
- Predicted probabilities: 0.5, 0.5  
- Sum \(p_i (1-p_i)\) = \(2 \times 0.5 \times 0.5 = 0.5\)

$$
SS_{left} = \frac{(-1)^2}{0.5 + 1} = \frac{1}{1.5} = 0.6667
$$

---

### Right Node (Samples \(X=3,4\))

- Residuals: +0.5, +0.5  
- Sum residuals = +1  
- Predicted probabilities: 0.5, 0.5  
- Sum \(p_i (1-p_i)\) = 0.5

$$
SS_{right} = \frac{(1)^2}{0.5 + 1} = 0.6667
$$

---

## 8️⃣ Calculate Gain for Split

$$
Gain = SS_{left} + SS_{right} - SS_{parent} = 0.6667 + 0.6667 - 0 = 1.3334
$$

---

## 9️⃣ Leaf Values (Output Score Update)

$$
\text{Leaf value} = \frac{\sum r_i}{\sum p_i (1-p_i) + \lambda}
$$

Left leaf:

$$
\frac{-1}{0.5 + 1} = -0.6667
$$

Right leaf:

$$
\frac{1}{0.5 + 1} = 0.6667
$$

---

## 🔟 Update Prediction (Learning Rate {eta = 0.3})

$$
z_{\text{new}} = z_{\text{old}} + \eta \times \text{leaf value}
$$

Since \( z_{\text{old}} = 0 \),

- Left node update:

$$
0 + 0.3 \times (-0.6667) = -0.2
$$

- Right node update:

$$
0 + 0.3 \times 0.6667 = 0.2
$$

---

## 1️⃣1️⃣ Convert Updated Log-Odds to Probability (Sigmoid)

- Left node:

$$
p = \frac{1}{1 + e^{0.2}} \approx 0.45
$$

- Right node:

$$
p = \frac{1}{1 + e^{-0.2}} \approx 0.55
$$

---

## 1️⃣2️⃣ Classification Decision

- Threshold = 0.5  
- Left node samples → predicted class 0  
- Right node samples → predicted class 1  

---




In [1]:
import pandas as pd
import numpy as np


np.random.seed(42)


X = np.random.uniform(0, 10, 200)

y = (X + np.random.normal(0, 1, 200)) > 5

y = y.astype(int)

df = pd.DataFrame({
    'X': X,
    'y': y
})


In [2]:
df

,X,y
0,3.745401,0
1,9.507143,1
2,7.319939,1
3,5.986585,1
4,1.560186,0
...,...,...
195,3.492096,0
196,7.259557,1
197,8.971103,1
198,8.870864,1


In [3]:
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier

In [4]:
x = df.iloc[: , 0]
y = df.iloc[: , 1]

In [7]:
x_train , x_test , y_train , y_test = train_test_split(x , y , test_size=0.2 , random_state=42)

In [8]:
x_train

79     1.158691
197    8.971103
38     6.842330
24     4.560700
122    3.180035
         ...   
106    4.103829
14     1.818250
92     7.607850
179    1.375209
102    3.143560
Name: X, Length: 160, dtype: float64

In [9]:
xgb = XGBClassifier()

In [10]:
xgb.fit(x_train , y_train)

,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [11]:
y_pred = xgb.predict(x_test)


In [12]:
from sklearn.metrics import r2_score
r2_score(y_test , y_pred)

0.6